# 🚀 ShopMind AI: Neural E-commerce Assistant

**Engineer:** Sriram Bahadur   
**Stack:** LangChain, Llama 3.1 (Groq), ChromaDB, HuggingFace Transformers

This project implements a robust Retrieval-Augmented Generation (RAG) pipeline designed to navigate the complexities of Amazon product reviews. It focuses on semantic search, low-level text preprocessing, and an agentic tool-calling architecture.

# ShopMind AI: End-to-End RAG + Agentic E-commerce Chatbot

## Project Goal

Create a smart shopping assistant that demonstrates:
- Data Collection & Preprocessing
- Feature Engineering
- Vector Database
- RAG Pipeline
- Prompt Engineering
- Agentic Workflow
- Evaluation Pipeline
- API/Backend + Frontend UI
- Deployment (ngrok + Streamlit/Gradio)
- Low-level ML practices (to showcase for Amazon ML Summer School)

## 2. Install Dependencies

This section installs all the necessary Python libraries required for our ShopMind AI chatbot. These include libraries for data handling (`datasets`), natural language processing (`nltk`, `transformers`), machine learning utilities (`scikit-learn`), building LLM applications (`langchain`, `langchain-community`), vector storage (`chromadb`, `sentence-transformers`), interacting with LLMs (`groq`), and creating the user interface (`gradio`). We also include `python-dotenv` for secure environment variable management.

In [ ]:
# Force-installing all specific LangChain components to ensure proper mapping
!pip install -U --force-reinstall -qqq langchain langchain-community langchain-chroma langchain-huggingface langchain-groq sentence-transformers chromadb
print("✅ Core LangChain components re-verified and fully loaded.")

## 3. NLTK Data & API Key Setup

Here we download essential NLTK data packages for text preprocessing (tokenization and stop words). We also set up placeholders for API keys required to access external services like Groq for our LLM. It's crucial to store API keys securely, typically using Colab's `userdata` secrets manager.

In [ ]:
import nltk
import os
from google.colab import userdata

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# --- API Key Setup ---
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✅ Groq API Key loaded successfully from Colab Secrets.")
except Exception as e:
    print("❌ Groq API Key not found in Colab secrets. Please add a secret named 'GROQ_API_KEY'.")

print("NLTK data ready.")

## 4. Dataset Loading & Exploration

We will use the "McAuley-Lab/Amazon-Reviews-2023" dataset from Hugging Face, specifically the "Electronics_v1_00" split. To manage computational resources and focus on demonstrating the RAG pipeline effectively, we'll sample a subset of this dataset (around 20,000 samples, as specified in the requirements).

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

# DATA INGESTION: Pulling from the McAuley-Lab Amazon 2023 Repository
# We're targeting 'Electronics' for high-density review patterns.

DATASET_ID = "McAuley-Lab/Amazon-Reviews-2023"
SUBSET = "Electronics_v1_00"

print(f"[*] Initializing stream from {DATASET_ID}...")

try:
    # Using streaming=True for memory efficiency in Colab
    raw_stream = load_dataset(DATASET_ID, SUBSET, split='train', streaming=True, trust_remote_code=True)

    # Taking 5000 samples for a balanced demonstration of the RAG bottleneck
    sample_list = []
    for i, entry in tqdm(enumerate(raw_stream), total=5000):
        if i >= 5000: break
        sample_list.append(entry)

    df = pd.DataFrame(sample_list)
    print(f"[+] Ingested {len(df)} records. Data mapping complete.")

except Exception as e:
    print(f"[!] Ingestion error: {e}. Falling back to cached local subset if available.")

display(df[['item_id', 'title', 'text', 'rating']].head())

In [ ]:
import pandas as pd
from datasets import Dataset
from IPython.display import display

print("Note: Remote dataset link returned 404. Generating high-quality synthetic data for ShopMind AI development...")

synthetic_data = [
    {"item_title": "Bose QuietComfort 35 II", "item_id": "B0756CYWWD", "rating": 5, "review_text": "The noise cancellation is world-class. Very comfortable for long flights.", "verified_purchase": True},
    {"item_title": "Sony WH-1000XM4", "item_id": "B0863TXGM3", "rating": 5, "review_text": "Incredible sound quality and the multi-point connection works perfectly.", "verified_purchase": True},
    {"item_title": "Anker Soundcore Life Q20", "item_id": "B07NM3RSRQ", "rating": 4, "review_text": "Great value for the price. Bass is strong but build feels a bit plastic.", "verified_purchase": True},
    {"item_title": "Apple AirPods Pro (2nd Gen)", "item_id": "B0BP9M1S9V", "rating": 5, "review_text": "Transparency mode is unmatched. Seamless integration with my iPhone.", "verified_purchase": False},
    {"item_title": "Samsung Galaxy Watch 5", "item_id": "B0B3S6HBSV", "rating": 4, "review_text": "Battery life is improved but still needs daily charging if used heavily.", "verified_purchase": True},
    {"item_title": "Logitech MX Master 3S", "item_id": "B09HM94VDS", "rating": 5, "review_text": "The scroll wheel is silent and the ergonomic shape prevents wrist pain.", "verified_purchase": True}
]

# Multiply the data to simulate a larger dataset for RAG testing
df = pd.DataFrame(synthetic_data * 100)

# Ensure the expected columns exist
if 'parent_id' not in df.columns:
    df['parent_id'] = df['item_id']

# Create the dataset_sample object for downstream cells
dataset_sample = Dataset.from_pandas(df.reset_index(drop=True))

print(f"Successfully initialized ShopMind AI with {len(df)} records.")
display(df.head())

## 5. Creating Rich Documents

To enrich the context for our chatbot, we combine product metadata (like title, ID, rating) and review text into a single 'rich document' for each entry. This process ensures that when the chatbot retrieves information, it gets a comprehensive view of the product and its associated reviews, which is vital for providing detailed and helpful responses.

In [ ]:
def create_rich_document(entry):
    product_info = f"Product Title: {entry.get('item_title', 'N/A')}\nProduct ID: {entry.get('item_id', 'N/A')}"
    review_info = f"Rating: {entry.get('rating', 'N/A')} out of 5 stars.\nReview: {entry.get('review_text', 'N/A')}"
    return f"{product_info}\n{review_info}"

print("Creating rich documents...")
# Applying the function to our synthetic dataset
dataset_sample = dataset_sample.map(lambda x: {"rich_document": create_rich_document(x)})
df = dataset_sample.to_pandas()

print("Rich documents created successfully.")
display(df[['item_title', 'rich_document']].head())

## 4. Data Cleaning & Preprocessing (Low-Level ML Practices)

Data cleaning is a critical step in any NLP pipeline to reduce noise and improve the quality of text data. Here, we'll perform several fundamental preprocessing steps using the NLTK library:

1.  **Tokenization**: Breaking down text into individual words or tokens. This is the first step to make text understandable for machine processing.
2.  **Lowercasing**: Converting all characters to lowercase to ensure that words like "The" and "the" are treated as the same word, reducing vocabulary size and improving consistency.
3.  **Stopword Removal**: Eliminating common words (e.g., "a", "an", "the", "is") that carry little semantic meaning and can add noise to our models. This helps focus on more significant terms.
4.  **Stemming**: Reducing words to their root or base form (e.g., "running", "ran", "runs" all become "run"). This helps in normalizing words and grouping together words with similar meanings, further reducing vocabulary size.

These low-level operations are fundamental to many traditional NLP and information retrieval tasks and are crucial for preparing data for machine learning models.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import string
import re

stop_words = set(stopwords.words('english'))
porter = PorterStemmer()

def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}0-9]', '', text)
    tokens = word_tokenize(text)
    return " ".join([porter.stem(w) for w in tokens if w not in stop_words])

print("Cleaning review text...")
df['cleaned_review_text'] = df['review_text'].apply(clean_text)
print("Cleaning complete.")

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import string
import re
from IPython.display import display

# Ensure all required NLTK resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Initialize NLTK components
stop_words = set(stopwords.words('english'))
porter = PorterStemmer()

def clean_text(text):
    """Performs tokenization, lowercasing, stopword removal, and stemming."""
    if not isinstance(text, str):
        return ""

    # 1. Lowercasing
    text = text.lower()

    # 2. Remove punctuation and numbers
    text = re.sub(f'[{re.escape(string.punctuation)}0-9]', '', text)

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Stopword Removal and Stemming
    cleaned_tokens = [porter.stem(word) for word in tokens if word not in stop_words and len(word) > 1]

    return " ".join(cleaned_tokens)

print("Applying text cleaning and preprocessing...")
df['cleaned_review_text'] = df['review_text'].apply(clean_text)

print("\n--- Original vs. Cleaned Review Text Example ---")
example_index = 0
print(f"Original Review:\n{df['review_text'].iloc[example_index]}")
print(f"\nCleaned Review:\n{df['cleaned_review_text'].iloc[example_index]}")

print(f"\nUpdated DataFrame head with 'cleaned_review_text':")
display(df[['review_text', 'cleaned_review_text']].head())

## 6. Feature Engineering

Feature engineering transforms raw data into a format that machine learning models can understand and learn from. For text data, this typically involves converting text into numerical representations. We will explore two key methods:

1.  **TF-IDF (Term Frequency-Inverse Document Frequency)**: A statistical measure that evaluates how relevant a word is to a document in a collection of documents. It increases proportionally to the number of times a word appears in the document but is offset by the frequency of the word in the corpus, which helps to adjust for the fact that some words appear more frequently in general.
    *   **Term Frequency (TF)**: The number of times a term appears in a document.
    *   **Inverse Document Frequency (IDF)**: A measure of how much information the word provides; words that are common across many documents (like "the" or "a") will have a low IDF.
    TF-IDF is useful for traditional information retrieval and for providing a sparse but interpretable representation of text.

2.  **Embedding Features (all-MiniLM-L6-v2)**: Modern NLP often relies on dense vector representations called embeddings. These embeddings capture semantic relationships between words and sentences. We will use `sentence-transformers` library with the `all-MiniLM-L6-v2` model, which is a state-of-the-art model for generating high-quality sentence embeddings. These dense vectors are particularly effective for tasks requiring semantic understanding, like similarity search in vector databases.
    *   **Low-level concept**: Embeddings are learned by neural networks, mapping words or sentences into a continuous vector space where words with similar meanings are located close to each other. This enables models to understand context and relationships far beyond what bag-of-words models like TF-IDF can achieve.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize the embedding model
print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ensure rich_document is ready for embedding
if 'rich_document' not in df.columns:
    print("Creating rich_document column...")
    def create_rich_doc(row):
        return f"Product: {row['item_title']}\nID: {row['item_id']}\nRating: {row['rating']}\nReview: {row['review_text']}"
    df['rich_document'] = df.apply(create_rich_doc, axis=1)

print("Generating embeddings for rich documents (this may take a minute)... module...")
# Generate embeddings for our synthetic dataset
embeddings = model.encode(df['rich_document'].tolist(), show_progress_bar=True)
df['embedding'] = list(embeddings)

print(f"Generated {len(embeddings)} embeddings successfully.")
display(df[['item_title', 'embedding']].head())

## 7. Vector Database Creation

A vector database is specialized for storing and querying high-dimensional vectors, such as the embeddings we generated. For our RAG system, the vector database allows us to perform efficient similarity searches, finding the most relevant documents (products and reviews) to a user's query.

We will use **ChromaDB**, an open-source vector database that can be run locally or in a client-server mode. It's well-suited for experimentation and smaller-scale deployments.

**Low-level concept**: When a user inputs a query, it will also be converted into an embedding. The vector database then calculates the 'distance' (e.g., cosine similarity) between the query embedding and all the document embeddings stored within it. Documents with the smallest distance (highest similarity) are considered the most relevant and are retrieved. This process is much faster and more scalable than traditional database lookups for semantic search.

## 8. RAG Pipeline + Prompt Engineering

The Retrieval-Augmented Generation (RAG) pipeline is central to our ShopMind AI chatbot. It combines the power of information retrieval (from our vector database) with the generative capabilities of an LLM. This allows the LLM to provide answers grounded in our specific product and review data, reducing hallucinations and improving relevance.

### How RAG Works:
1.  **User Query**: The user asks a question.
2.  **Retrieval**: The query is embedded, and relevant documents (product reviews, metadata) are retrieved from the vector database based on semantic similarity.
3.  **Augmentation**: The retrieved documents are passed to the LLM along with the user's original query.
4.  **Generation**: The LLM uses this augmented context to generate a more accurate and informed response.

### Prompt Engineering:
We will use a carefully crafted system prompt to guide the LLM's behavior, ensuring it acts as a helpful e-commerce shopping assistant, utilizes the provided context, and adheres to specific output formats.

We will use `Groq` as our LLM provider for its speed, specifically leveraging the `Llama-3.1-70B-versatile` model. For the RAG implementation, we'll use LangChain's `create_retrieval_chain` for a streamlined setup.

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

print("Initializing ShopMind AI RAG Chain...")

# 1. Initialize the LLM with Groq
llm = ChatGroq(
    temperature=0.1,
    model_name="llama-3.1-70b-versatile",
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

# 2. Define the System Prompt
system_prompt = """
You are ShopMind AI, an expert e-commerce assistant.
Your goal is to help users make informed shopping decisions using the provided context.
Context contains product details and customer reviews.

Rules:
- Be professional, enthusiastic, and helpful.
- Use the context to answer. If the context doesn't contain the answer, say you don't have enough info.
- Mention if a review is from a 'Verified Purchase' if that info is available.
- Summarize pros and cons clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 3. Create the Chains
# The retriever pulls top 3 relevant documents from ChromaDB
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

print("✅ RAG Chain successfully initialized.")

# 4. Final Test Run
test_query = "Which headphones have the best noise cancellation according to the reviews?"
print(f"\nTesting Query: {test_query}")
try:
    result = rag_chain.invoke({"input": test_query})
    print(f"ShopMind AI Response: {result['answer']}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# STEP 2: Restore data and initialize the RAG pipeline
import os
import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 1. Restore Data State
print("Restoring synthetic product data...")
synthetic_data = [
    {"item_title": "Bose QuietComfort 35 II", "item_id": "B0756CYWWD", "rating": 5, "review_text": "The noise cancellation is world-class. Very comfortable for long flights."},
    {"item_title": "Sony WH-1000XM4", "item_id": "B0863TXGM3", "rating": 5, "review_text": "Incredible sound quality and the multi-point connection works perfectly."},
    {"item_title": "Anker Soundcore Life Q20", "item_id": "B07NM3RSRQ", "rating": 4, "review_text": "Great value for the price. Bass is strong but build feels a bit plastic."},
    {"item_title": "Apple AirPods Pro (2nd Gen)", "item_id": "B0BP9M1S9V", "rating": 5, "review_text": "Transparency mode is unmatched. Seamless integration with my iPhone."}
]
df = pd.DataFrame(synthetic_data * 10)
def create_rich_doc(row): return f"Product: {row['item_title']}\nRating: {row['rating']}\nReview: {row['review_text']}"
df['rich_document'] = df.apply(create_rich_doc, axis=1)

# 2. Re-initialize Vector Store
print("Initializing Vector Store...")
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
langchain_documents = [Document(page_content=row['rich_document'], metadata={"item_title": row['item_title']}) for _, row in df.iterrows()]
vector_store = Chroma.from_documents(documents=langchain_documents, embedding=embedding_function)

# 3. Finalize and Launch RAG Chain
print("Launching ShopMind AI...")
llm = ChatGroq(temperature=0.1, model_name="llama-3.1-70b-versatile", groq_api_key=os.environ.get("GROQ_API_KEY"))
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are ShopMind AI, a helpful e-commerce assistant. Use this context: {context}"),
    ("human", "{input}")
])
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), document_chain)

print("\u2705 ShopMind AI is LIVE!")
test_res = rag_chain.invoke({"input": "Which headphones are best for noise cancellation?"})
print(f"\nTest Response: {test_res['answer']}")

## 9. Agentic Workflow

An agentic workflow enhances the chatbot's capabilities by allowing it to dynamically choose and use tools based on the user's intent. Instead of a fixed, linear response, the agent can reason through a problem, decide which tools are best suited, execute them, and then use the results to formulate a comprehensive answer. This enables multi-step reasoning and more sophisticated interactions.

Our ShopMind AI will be equipped with several tools:
-   **Retrieval Tool**: To search our ChromaDB for relevant product and review information.
-   **Summarizer Tool**: To condense long review texts or product descriptions into key takeaways.
-   **Calculator Tool**: For simple arithmetic operations, useful for comparing prices or other numerical attributes.

We will use LangChain's `create_openai_functions_agent` (or a similar construct for Groq) to build this agent, allowing the LLM to decide when and how to use these tools.

In [ ]:
from langchain.agents import AgentExecutor # Corrected import path for AgentExecutor
from langchain.agents import create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate # Ensure ChatPromptTemplate is imported for agent_prompt

# --- 9.1. Define Tools for the Agent ---
print("Defining tools for the agent...")

# 1. Retrieval Tool (using our existing RAG pipeline)
# The RAG chain already handles retrieval and initial generation.
# We need to expose it as a tool.
@tool
def product_search(query: str) -> str:
    """Searches for product and review information using the ShopMind AI RAG system.
    Input should be a detailed query about a product or reviews.
    """
    print(f"\n[Agent Tool Call: product_search] Query: {query}")
    # Invoke the RAG chain directly
    response = rag_chain.invoke({"input": query})
    return response['answer']

# 2. Summarizer Tool (simple placeholder, can be enhanced with another LLM call)
@tool
def summarize_text(text: str) -> str:
    """Summarizes a given text into its key points. Useful for condensing long reviews or descriptions.
    Input should be the text to summarize.
    """
    print(f"\n[Agent Tool Call: summarize_text] Summarizing text (first 100 chars): {text[:100]}...")
    # In a real scenario, this would use an LLM or another summarization model.
    # For now, a very basic summarization.
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert summarizer. Condense the following text into 2-3 key sentences."),
        ("human", "{text}")
    ])
    summarizer_chain = summary_prompt | llm
    return summarizer_chain.invoke({"text": text}).content

# 3. Calculator Tool
@tool
def calculator(expression: str) -> str:
    """Executes a Python arithmetic expression and returns the result. Useful for calculations.
    Input should be a valid mathematical expression string (e.g., '2 + 2 * 3').
    """
    print(f"\n[Agent Tool Call: calculator] Calculating: {expression}")
    try:
        return str(eval(expression)) # eval is generally unsafe, but acceptable for simple math in this context
    except Exception as e:
        return f"Error during calculation: {e}"

# List all tools available to the agent
tools = [product_search, summarize_text, calculator]
print(f"Agent has {len(tools)} tools available: {[t.name for t in tools]}")

# --- 9.2. Construct the Agent ---
print("Constructing the LangChain Agent...")

# Placeholder for the system prompt from earlier. It will be used by the agent.
# REMINDER: Replace `system_prompt_template` with the actual prompt from the requirements.
agent_system_message = system_prompt_template

# Create the agent using our LLM and tools.
# The prompt passed to create_tool_calling_agent needs specific placeholders.
# The ChatPromptTemplate defines how the agent's internal reasoning works.

# This defines the agent's behavior and access to tools.
agent_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', agent_system_message + "\nYou have access to the following tools: {tools}"),
        ('human', '{input}'),
        ('placeholder', '{agent_scratchpad}') # This is where the agent's thoughts and tool outputs go
    ]
)

# LangChain's create_tool_calling_agent is designed for models that support tool calling (like Llama 3.1 70B)
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# Create the AgentExecutor to run the agent
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # Set to True to see agent's thought process
    handle_parsing_errors=True # Gracefully handle JSON parsing errors from LLM
)

print("LangChain AgentExecutor created.")

# --- Test the Agentic Workflow ---
print("\n--- Testing the Agentic Workflow ---")

# Test 1: Simple search
print("\nTest 1: Simple product search")
agent_query_1 = "Tell me about the 'Bose QuietComfort 35 II' headphones."
result_1 = agent_executor.invoke({"input": agent_query_1})
print(f"Agent Response: {result_1['output']}")

# Test 2: Query requiring summarization
print("\nTest 2: Query requiring summarization (Note: summarizer will be called internally)")
agent_query_2 = "What are the main takeaways from reviews for 'Sony WH-1000XM4' headphones regarding battery life and sound quality?"
result_2 = agent_executor.invoke({"input": agent_query_2})
print(f"Agent Response: {result_2['output']}")

# Test 3: Query requiring calculation
print("\nTest 3: Query requiring calculation")
agent_query_3 = "What is 15% of 250 dollars?"
result_3 = agent_executor.invoke({"input": agent_query_3})
print(f"Agent Response: {result_3['output']}")

print("\n--- Agentic Workflow Tests Complete ---")

## 10. Evaluation Pipeline

Evaluating RAG systems is crucial to ensure they provide accurate, relevant, and faithful responses. A comprehensive evaluation pipeline typically involves assessing both the **retrieval** component and the **generation** component.

### 10.1. Retrieval Metrics
Retrieval metrics assess how effectively the system fetches relevant documents for a given query.

-   **Hit Rate**: Measures whether the *most relevant* document (or at least one relevant document) is among the top-k retrieved documents. A higher hit rate indicates better recall.
-   **MRR (Mean Reciprocal Rank)**: Measures the average of the reciprocal ranks of the first relevant document. It emphasizes getting relevant documents in higher ranks. If the first relevant document is at rank 1, reciprocal rank is 1; if at rank 2, it's 1/2, etc.

### 10.2. Generation Metrics
Generation metrics assess the quality of the LLM's response based on the retrieved context.

-   **Faithfulness**: Measures how factually accurate the generated answer is with respect to the retrieved context. It checks if the answer can be solely inferred from the given documents.
-   **Answer Relevance**: Assesses how relevant the generated answer is to the original user query. It also checks for conciseness and completeness.

We will use the `ragas` library for some of these evaluations, as it provides a robust framework for RAG evaluation. We will need to define a small set of example questions and their expected answers for this evaluation.

## 11. Gradio/Streamlit UI

To make our ShopMind AI accessible and interactive, we'll build a user-friendly web interface using **Gradio**. Gradio allows us to quickly create customizable UI components for machine learning models and share them easily. This interface will enable users to input queries and receive responses from our agentic RAG chatbot.

### UI Features:
-   A chat interface to interact with the ShopMind AI.
-   Clear display of conversation history.
-   An intuitive way for users to ask questions about products, comparisons, and reviews.

In [ ]:
import gradio as gr
import os

print("Setting up Gradio UI for ShopMind AI...")

# Define the chat function that will be called by Gradio
def chat_with_shopmind_ai(message, history):
    """Processes user messages using the AgentExecutor and returns the response.

    Args:
        message (str): The user's input message.
        history (list): List of previous chat messages (list of tuples).

    Returns:
        str: The AI's response.
    """
    print(f"\n[User Message]: {message}")

    # Convert Gradio chat history format to LangChain's message format if needed
    # For this simple agent invocation, we only pass the current message.
    # More complex agents might maintain a full conversation history.

    try:
        # Invoke the AgentExecutor with the user's message
        # The AgentExecutor will decide whether to use tools or directly generate a response.
        response = agent_executor.invoke({"input": message})
        ai_response = response['output']
    except Exception as e:
        ai_response = f"I encountered an error: {e}. Please try again or rephrase your query."
        print(f"[Error in chat_with_shopmind_ai]: {e}")

    print(f"[ShopMind AI Response]: {ai_response}")
    return ai_response


# Create the Gradio interface
# The ChatInterface automatically handles chat history, input, and output.
iface = gr.ChatInterface(
    fn=chat_with_shopmind_ai,
    chatbot=gr.Chatbot(height=500, type='messages'), # Added type='messages' to address UserWarning
    textbox=gr.Textbox(placeholder="Ask me anything about products, reviews, or comparisons!", container=False, scale=7),
    title="ShopMind AI: Your E-commerce Shopping Assistant",
    description="Ask me questions about product features, reviews, compare items, or get recommendations.",
    theme="soft",
    examples=[
        "What are the main features of the latest iPhone?",
        "Summarize reviews for the Samsung Galaxy Watch 5 regarding battery life.",
        "Compare the comfort of Sony WH-1000XM4 and Bose QuietComfort 35 II headphones.",
        "What is 15% of 250 dollars?"
    ],
    cache_examples=False # Set to True to cache example outputs if they don't change often
    # clear_btn="Clear Chat", # Removed for broader Gradio compatibility
    # undo_btn="Undo Last Turn", # Removed for broader Gradio compatibility
    # submit_btn="Send" # Removed for broader Gradio compatibility
)

print("Gradio UI interface created.")
print("To launch the Gradio UI, run `iface.launch()`.")

## 12. Deployment with ngrok

To make our Gradio application publicly accessible from a Colab environment, we'll use `ngrok`. `ngrok` creates a secure tunnel to your local development server, allowing it to be exposed to the internet. This is particularly useful for sharing demos or testing applications without needing a dedicated server.

### How ngrok works:
1.  **Install ngrok**: We'll install the `pyngrok` library.
2.  **Authenticate**: You'll need an ngrok authentication token (available for free from their website). This token associates your tunnel with your ngrok account.
3.  **Launch Gradio with Share**: When Gradio is launched with `share=True`, it attempts to create a public link. `pyngrok` integrates with Gradio to handle the tunneling.

**Note**: The ngrok tunnel remains active only as long as your Colab runtime is running. If the runtime disconnects, the public URL will become inactive.

In [ ]:
!pip install -qqq pyngrok

from pyngrok import ngrok
import getpass
import os

print("Setting up ngrok for public access...")

# --- Ngrok Authentication ---
# Get your ngrok authentication token from https://dashboard.ngrok.com/get-started/your-authtoken
# You only need to run this cell once per Colab session if your token is saved.

# Securely get ngrok token from Colab secrets or prompt the user
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("Please enter your ngrok Authtoken. You can find it on your ngrok dashboard: https://dashboard.ngrok.com/get-started/your-authtoken")
    NGROK_AUTH_TOKEN = getpass.getpass("Enter your ngrok Authtoken: ")
    os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN # Set for current session

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("Ngrok Authtoken set successfully.")
except Exception as e:
    print(f"Error setting ngrok authtoken: {e}. Please check your token.")

print("Launching Gradio UI with ngrok tunnel...")
# Launch the Gradio interface with share=True to create a public link via ngrok
# This will print a public URL once the server starts.
# iface.launch(share=True, debug=True) # debug=True can provide more verbose output

# Due to Colab execution model, we usually run launch() in a separate cell if we want to continue executing other cells.
# For a single execution flow for the entire notebook demonstration, it's often the final step.
# As requested by the user, we'll indicate this is the final launch step.

print("Run the following line in a new cell to launch the Gradio UI:")
print("iface.launch(share=True)")

# You can manually copy paste the below line to a new cell and execute it
# iface.launch(share=True)
